# Data Audit & Cleaning

This notebook audits the raw BMW global sales dataset (2018-2025), identifies data quality issues, and applies the necessary fixes before the data is used in `02_Analysis.ipynb`.

## 1. Audit

This section reviews the dataset's structure, data types, value ranges, categorical consistency, and outliers, in order to build a full list of cleaning actions needed before analysis.

In [222]:
import pandas as pd

### 1.a Data overview

#### First look at the dataset structure and content

In [223]:
df = pd.read_csv("/Users/thibautraiola/DATA-ANALYST/Project4Portfolio/BMW-Sales-Global/bmw_global_sales_DIRTY.csv")
df.head(10)

,Year,Month,Region,Model,Units_Sold,Avg_Price_EUR,Revenue_EUR,BEV_Share,Premium_Share,GDP_Growth,Fuel_Price_Index
0,2021,12,Europe,X3,9715.0,58256.0,565957040.0,0.078,19.94,2.80,1.20
1,2024,4,Europe,3 Series,12042.0,47648.0,573777216.0,0.151,16.13,3.25,1.23
2,2022,9,USA,MINI,8458.0,40122.0,339351876.0,0.119,18.12,3.76,1.22
3,2019,4,RestOfWorld,MINI,6710.0,43668.0,293012280.0,0.045,6.62,3.44,1.08
4,2022,1,RestOfWorld,iX,6091.0,75460.0,459626860.0,0.126,5.55,2.50,1.20
5,2020,1,USA,iX,11133.0,73946.0,823240818.0,0.076,14.91,2.96,1.06
6,2018,12,Europe,X3,12695.0,56159.0,712938505.0,0.012,20.78,2.50,0.99
7,2022,11,China,3 Series,3861.0,47092.0,181822212.0,0.127,19.33,1.18,1.18
8,2025,6,China,i4,9116.0,65168.0,594071488.0,0.196,16.87,2.17,1.31
9,2019,1,RestOfWorld,3 Series,9032.0,49731.0,449170392.0,0.069,6.50,3.26,1.06


#### Checking data types and null count per column

In [224]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3087 entries, 0 to 3086
Data columns (total 11 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Year              3087 non-null   int64  
 1   Month             3087 non-null   int64  
 2   Region            3087 non-null   object 
 3   Model             3087 non-null   object 
 4   Units_Sold        3057 non-null   object 
 5   Avg_Price_EUR     3057 non-null   float64
 6   Revenue_EUR       3087 non-null   float64
 7   BEV_Share         3087 non-null   float64
 8   Premium_Share     3087 non-null   float64
 9   GDP_Growth        3087 non-null   float64
 10  Fuel_Price_Index  3087 non-null   float64
dtypes: float64(6), int64(2), object(3)
memory usage: 265.4+ KB


**Dtype:** `Units_Sold` is typed as `object` but should be integer, car sales cannot be fractional.

All other columns have the expected data type: floats for continuous values, integers for discrete ones.

#### Statistical overview of numerical columns

In [225]:
df.describe()

,Year,Month,Avg_Price_EUR,Revenue_EUR,BEV_Share,Premium_Share,GDP_Growth,Fuel_Price_Index
count,3087.000000,3087.000000,3057.000000,3.087000e+03,3087.000000,3087.000000,3087.000000,3087.000000
mean,2021.500486,6.509232,63851.187112,5.777879e+08,0.172336,14.655121,3.057606,1.176184
std,2.294132,3.468191,14668.652384,1.907300e+09,2.518211,5.337963,1.018062,0.118282
min,2018.000000,0.000000,40011.000000,-8.517752e+08,-0.500000,5.090000,0.100000,0.920000
25%,2019.000000,3.000000,49998.000000,3.113796e+08,0.056000,6.950000,2.370000,1.080000
50%,2021.000000,7.000000,63499.000000,4.811743e+08,0.107000,16.250000,3.050000,1.180000
75%,2024.000000,10.000000,70970.000000,6.737491e+08,0.158000,18.930000,3.850000,1.270000
max,2025.000000,13.000000,93994.000000,8.368180e+10,99.000000,20.970000,5.820000,1.410000


* `Year`: range correct (2018-2025).
* `Month`: values between 0 and 13 found, impossible (valid range: 1-12).
* `Avg_Price_EUR`: valid, min and max within normal range for the automotive industry.
* `Revenue_EUR`: negative values found, invalid for a revenue column. Max value (e+10) is suspiciously high compared to the distribution mean (e+8), further analysis required.
* `BEV_Share`: values outside the 0-1 range found (-0.5 and 99.0), invalid for a proportion.
* `Premium_Share`: valid, range 0-100 as expected for a market share percentage.
* `GDP_Growth`: valid, within expected range.
* `Fuel_Price_Index`: valid, within expected range.

### 1.b Data types verification

We investigate the root cause of the `Units_Sold` type issue identified in the previous step.

In [226]:
mask = pd.to_numeric(df['Units_Sold'], errors="coerce").isnull()
df[mask]

,Year,Month,Region,Model,Units_Sold,Avg_Price_EUR,Revenue_EUR,BEV_Share,Premium_Share,GDP_Growth,Fuel_Price_Index
26,2021,3,USA,i4,NaN,65722.0,276952508.0,0.073,19.64,1.71,1.11
108,2025,9,RestOfWorld,i4,NaN,65479.0,244498586.0,0.217,6.64,3.40,1.35
122,2022,7,China,X5,NaN,68723.0,249945551.0,0.101,14.63,2.69,1.18
130,2021,11,RestOfWorld,iX,NaN,76106.0,211955210.0,0.097,5.49,1.86,1.17
383,2020,2,USA,5 Series,NaN,61309.0,197231053.0,0.065,18.43,3.08,1.13
418,2021,1,USA,i4,NaN,64351.0,375037628.0,0.089,18.71,2.47,1.16
566,2018,9,RestOfWorld,X3,"6,792",58143.0,394907256.0,0.016,5.51,4.22,0.93
571,2020,2,USA,X5,"8,704",69958.0,608914432.0,0.066,18.43,3.08,1.13
585,2019,11,Europe,5 Series,"9,851",60550.0,596478050.0,0.047,14.48,2.40,1.01
636,2023,2,China,X7,NaN,92513.0,849546879.0,0.139,16.93,2.53,1.29


The `Units_Sold` column is typed as `object` instead of numeric. Two issues explain this: missing values (NaN) and numeric values formatted with a thousands separator (9,230 instead of 9230), likely caused by an Excel export. We will fix this data type problem later in the cleaning section.

### 1.c Data understanding & duplicate verification

Checking categorical columns for inconsistencies and duplicates caused by formatting differences.

In [227]:
df['Region'].unique()

array(['Europe', 'USA', 'RestOfWorld', 'China', 'CHINA', 'china', 'usa',
       'EUROPE', 'restofworld', 'europe', 'RESTOFWORLD'], dtype=object)

Inconsistencies found in the `Region` column — values must follow a single format: Title Case.

In [228]:
df['Model'].unique()

array(['X3', '3 Series', 'MINI', 'iX', 'i4', 'X7', 'X5', '5 Series'],
      dtype=object)

No inconsistencies found — 8 models identified.

### 1.d Outliers detection

Here we highlight potential outliers by verifying the maximum value of `Revenue_EUR`. During the data overview phase, we noticed its max value being surprisingly high compared to Q3 and median.

The approach used here could be applied to any column; we focus on Revenue_EUR as other columns don't show any alarming values based on domain knowledge.

In [229]:
Revenue_Q1 = df['Revenue_EUR'].quantile(0.25)
Revenue_Q3 = df['Revenue_EUR'].quantile(0.75)
Revenue_IQR = Revenue_Q3 - Revenue_Q1
outliers = df[df['Revenue_EUR'] > (Revenue_Q3 + 1.5 * Revenue_IQR)].copy()
print(len(outliers))
outliers.sort_values('Revenue_EUR', ascending=False)

31


,Year,Month,Region,Model,Units_Sold,Avg_Price_EUR,Revenue_EUR,BEV_Share,Premium_Share,GDP_Growth,Fuel_Price_Index
1780,2023,2,RestOfWorld,X5,11938.0,70097.0,8.368180e+10,0.137,6.42,2.26,1.27
1531,2022,9,USA,X5,6843.0,67688.0,4.631890e+10,0.118,18.12,3.76,1.22
2957,2022,2,Europe,X3,5938.0,57763.0,3.429967e+10,0.123,19.66,3.28,1.23
2947,2025,5,RestOfWorld,X3,4772.0,59810.0,2.854133e+10,0.194,5.29,2.31,1.38
2617,2023,5,USA,iX,12917.0,73597.0,9.506524e+09,0.152,15.80,5.17,1.27
1078,2023,3,USA,i4,7062.0,64260.0,4.538041e+09,0.156,16.61,1.95,1.23
2494,2019,11,China,5 Series,5328.0,61449.0,3.274003e+09,0.038,14.29,2.81,1.02
731,2020,8,China,5 Series,3891.0,63198.0,2.459034e+09,0.069,17.49,3.65,1.07
626,2018,13,China,MINI,4624.0,42828.0,1.980367e+09,0.031,15.82,1.25,0.96
1621,2025,3,China,X7,15448.0,92794.0,1.433482e+09,0.205,18.76,4.46,1.34


31 outliers found, representing 1% of total values, a common situation. However, their `Revenue_EUR` distribution is very wide: from 1.223837e+09 to 8.368180e+10.

To better compare the magnitude of these outliers, we express each one's `Revenue_EUR` as a percentage of the group's median.

In [ ]:
outliers_Rev_median = outliers['Revenue_EUR'].median()
outliers['Rev_median_percent'] = (outliers['Revenue_EUR']/ outliers_Rev_median) * 100
outliers.sort_values('Rev_median_percent', ascending=False)

The four highest `Revenue_EUR` values are more than 20 times higher than the outliers' median, and the highest is 63 times higher.

These values are extremely high and need further analysis. This will be addressed in section 2.c using a revenue validation hypothesis.

## 2. Cleaning

This section applies the fixes identified during the audit: correcting data types, removing invalid values, standardizing categorical formatting, and validating revenue outliers against a reconstruction hypothesis.

In [231]:
#Creating a copy of the initial dataframe
df_cleaning = df.copy(deep=True)

### 2.a Data types fixing

* Fixing the thousands separator issue in `Units_Sold` column.
* `Int64` is used instead of `float64` as car sales are whole numbers — successful conversion to Int64 confirms no decimal values were present in the column.

In [232]:
df_cleaning['Units_Sold'] = pd.to_numeric(df_cleaning['Units_Sold'].str.replace(",", ""), errors="coerce")
df_cleaning['Units_Sold'] = df_cleaning['Units_Sold'].astype("Int64")

### 2.b Data inconsistency fixing

Fixing invalid values identified during the audit phase:

* Removing `Month` values outside the valid 1-12 range
* Removing negative `Revenue_EUR` values
* Removing `BEV_Share` values outside the 0-1 range
* Standardizing `Region` formatting to Title Case

In [ ]:
df_cleaning = df_cleaning[df_cleaning['Month'].between(1,12)]
df_cleaning = df_cleaning[df_cleaning['Revenue_EUR'] >= 0]
df_cleaning = df_cleaning[df_cleaning['BEV_Share'].between(0,1)]
df_cleaning['Region'] = df_cleaning['Region'].str.title()
df_cleaning['Region'].unique()

### 2.c Outliers

Abnormally high values were identified in the `Revenue_EUR` column. These could be legitimate outliers or incorrect data entries. To determine which category they fall into, we establish the following hypothesis:

Revenue_EUR = Units_Sold × Avg_Price_EUR

Two columns are added to test it: `Estimated_Revenue`, computed from the formula above, and `Revenue_Match`, a boolean comparing `Estimated_Revenue` to the original `Revenue_EUR`.

In [ ]:
df_cleaning['Estimated_Revenue'] = (df_cleaning['Units_Sold'] * df_cleaning['Avg_Price_EUR'])
df_cleaning['Revenue_Match'] = (df_cleaning['Estimated_Revenue'] == df_cleaning['Revenue_EUR'])
print(len(df_cleaning[~df_cleaning['Revenue_Match']]))
df_cleaning[~df_cleaning['Revenue_Match']]

We found only 15 observations, representing less than 0.5%, where the hypothesis is incorrect.

In [238]:
df_cleaning = df_cleaning[df_cleaning['Revenue_Match']]

Observations not matching the hypothesis have been removed — with 99.4% accuracy, the hypothesis is validated.

In [239]:
# Verify top Revenue_EUR values after removing non-matching rows
df_cleaning.sort_values('Revenue_EUR', ascending=False).head(10)

,Year,Month,Region,Model,Units_Sold,Avg_Price_EUR,Revenue_EUR,BEV_Share,Premium_Share,GDP_Growth,Fuel_Price_Index,Estimated_Revenue,Revenue_Match
1621,2025,3,China,X7,15448,92794.0,1.433482e+09,0.205,18.76,4.46,1.34,1433481712.0,True
19,2024,12,China,X7,14810,92812.0,1.374546e+09,0.173,19.13,1.86,1.27,1374545720.0,True
40,2025,6,Restofworld,X7,14786,92278.0,1.364423e+09,0.198,5.31,3.12,1.37,1364422508.0,True
1334,2025,12,Europe,X7,14561,93587.0,1.362720e+09,0.197,20.15,2.50,1.35,1362720307.0,True
2927,2024,9,China,X7,15061,90157.0,1.357855e+09,0.160,19.75,2.40,1.31,1357854577.0,True
391,2025,9,China,X7,14802,90096.0,1.333601e+09,0.204,20.90,5.43,1.33,1333600992.0,True
2314,2021,9,Usa,X7,14215,93386.0,1.327482e+09,0.092,18.52,3.18,1.11,1327481990.0,True
2576,2021,12,China,X7,14353,91024.0,1.306467e+09,0.095,16.14,3.31,1.12,1306467472.0,True
1870,2022,6,Europe,X7,14100,91334.0,1.287809e+09,0.112,20.37,3.07,1.12,1287809400.0,True
2572,2022,12,Europe,X7,13906,90555.0,1.259258e+09,0.117,14.24,1.31,1.22,1259257830.0,True


In [240]:
df_cleaning = df_cleaning.drop(['Estimated_Revenue', 'Revenue_Match'], axis=1)

## Data Cleaning Summary

Initial dataset: 3,087 rows × 11 columns. Final dataset: 2,975 rows × 11 columns (112 rows removed).

| Column | Actions |
|---|---|
| `Units_Sold` | Converted from `object` to `Int64`, thousands separator removed |
| `Month` | 4 observations outside the 1-12 range removed |
| `Revenue_EUR` | Negative values removed |
| `BEV_Share` | Values outside the 0-1 range removed |
| `Region` | Inconsistent casing standardized to Title Case |
| `Estimated_Revenue`, `Revenue_Match` | Temporary columns used for outlier validation, then dropped |
| *(no column, row-level check)* | 15 observations where `Revenue_EUR` ≠ `Units_Sold` × `Avg_Price_EUR` removed |

## Dataframe export

In [241]:
df_cleaning = df_cleaning.reset_index(drop=True)

In [242]:
df_cleaning.info()
df_cleaning.describe()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2975 entries, 0 to 2974
Data columns (total 11 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Year              2975 non-null   int64  
 1   Month             2975 non-null   int64  
 2   Region            2975 non-null   object 
 3   Model             2975 non-null   object 
 4   Units_Sold        2975 non-null   Int64  
 5   Avg_Price_EUR     2975 non-null   float64
 6   Revenue_EUR       2975 non-null   float64
 7   BEV_Share         2975 non-null   float64
 8   Premium_Share     2975 non-null   float64
 9   GDP_Growth        2975 non-null   float64
 10  Fuel_Price_Index  2975 non-null   float64
dtypes: Int64(1), float64(6), int64(2), object(2)
memory usage: 258.7+ KB


,Year,Month,Units_Sold,Avg_Price_EUR,Revenue_EUR,BEV_Share,Premium_Share,GDP_Growth,Fuel_Price_Index
count,2975.000000,2975.000000,2975.0,2975.000000,2.975000e+03,2975.000000,2975.000000,2975.000000,2975.000000
mean,2021.517311,6.502521,8002.226218,63788.606723,5.122532e+08,0.108127,14.670837,3.059728,1.177055
std,2.289185,3.455342,3180.517925,14664.485024,2.434342e+08,0.057970,5.333991,1.022109,0.117966
min,2018.000000,1.000000,2379.0,40011.000000,1.045314e+08,-0.000000,5.090000,0.100000,0.920000
25%,2020.000000,3.500000,5229.0,49966.500000,3.126872e+08,0.057000,14.050000,2.370000,1.080000
50%,2022.000000,7.000000,8025.0,63463.000000,4.820672e+08,0.108000,16.270000,3.050000,1.180000
75%,2024.000000,10.000000,10563.0,70937.000000,6.735569e+08,0.158500,19.005000,3.870000,1.270000
max,2025.000000,12.000000,15914.0,93994.000000,1.433482e+09,0.223000,20.970000,5.820000,1.410000


In [243]:
df_cleaning.to_csv("bmw_data_cleaned.csv",index=False)